# Tarea para el Hogar 02

Esta Tarea para el Hogar 02 se entrega el final de la segunda clase
<br> se espera de usted que intente avanzar con los desafios propuestos y que los traiga terminados para la Clase 03, ya que se analizarán los resultados

##  1. Ensembles de Modelos

Vea el siguiente video [BBC - The Code - The Wisdom of the Crowd](https://www.youtube.com/watch?v=iOucwX7Z1HU)    ( 5 min)


Lea los siguientes artículos


*   [The Wisdom of Crowds (Vox Populi) by Francis Galton](https://www.all-about-psychology.com/the-wisdom-of-crowds.html)  (10 min)
*   [A Gentle Introduction to Ensemble Learning](https://machinelearningmastery.com/what-is-ensemble-learning/)  (10 min)





---



##  2.  Zero2Hero   primera parte
Se han lanzado los primeros fascículos coleccionables llamados "from Zero to Hero" que muy detalladamente, paso a paso enseñan todo lo necesario de R para entender los scripts oficiales de la asignatura.
Están en el repositorio oficial de la asignatura, carpeta  **src/zero2hero**



---



## 3.  Grid Search

Busque en internet el precido significado de los hiperparámetros de la librería **rpart**  que está implementando el algoritmo **CART**  Classification and Regression Trees  propuesto en el año 1984 por Leo Brieman:

*   cp
*   maxdepth
*   minsplit
*   minbucket

Entienda que valores es razonable tome cada hiperparámetro,  en particular profundice en el hiperparámetro  **cp**  y la posibilidad que tome valores negativos.  Es válido consultar a su amigo de *capacidades especiales*  ChatGPT


En las siguientes celdas a un notebook incompleto, un esqueleto de codigo brindado a modo de facilitarle la tarea de codeo y permitir que su valiosa cognición se concentre temas conceptuales de Ciencia de Datos

Modifiquelo agregando loops para que recorra TODOS los hiperparámetros de rpart  < cp, maxdepth, minsplit, minbucket >, y luego póngalo a correr. Recuerde cambiar por SU semilla
Tenga muy presente la granularidad que eligirá para cada hiperparámetro.

### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Tipe -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# # primero establecer el Runtime de Python 3
# from google.colab import drive
# drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
# %%shell

# mkdir -p "/content/.drive/My Drive/dm"
# mkdir -p "/content/buckets"
# ln -sfn "/content/.drive/My Drive/dm"   /content/buckets/b1

# mkdir -p ~/.kaggle
# cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
# chmod 600 ~/.kaggle/kaggle.json


# mkdir -p /content/buckets/b1/exp
# mkdir -p /content/buckets/b1/datasets
# mkdir -p /content/datasets


# # defino funcion descargar()
# descargar() {
#   carpeta_destino="/content/buckets/b1/datasets/"
#   url_origen="https://storage.googleapis.com/open-courses/itba2026-7c9a/dm/"
#   archivo="$1"

#   if ! test -f "$carpeta_destino""$archivo"; then
#     wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
#   fi

#   if ! test -f  "/content/datasets/""$archivo"; then
#     cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
#   fi;
# }


# # hago la descarga efectiva, llamando a descargar()
# descargar  "dataset_pequeno.csv"


limpio el ambiente de R

In [3]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,672856,36.0,1489770,79.6,NA,1474555,78.8
Vcells,1250826,9.6,8388608,64.0,49152,2014427,15.4


In [4]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")
if (!require("primes")) install.packages("primes")
require("primes")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart

Loading required package: parallel

Loading required package: primes



Aqui debe poner SU semiila primigenia

In [5]:
# Seteo local: correr este notebook desde la raiz del repo o desde src/arboles
if (basename(getwd()) == "arboles") {
  ROOT_DIR <- normalizePath(file.path(getwd(), "../.."), mustWork = TRUE)
} else {
  ROOT_DIR <- normalizePath(getwd(), mustWork = TRUE)
}

DATA_DIR <- file.path(ROOT_DIR, "datasets")
EXP_DIR <- file.path(ROOT_DIR, "exp")
KAGGLE_JSON <- file.path(ROOT_DIR, "kaggle.json")

In [6]:
DATA_DIR
EXP_DIR
KAGGLE_JSON


[1] "/Users/selewaut/Code Projects/master/dm2026b/datasets"

[1] "/Users/selewaut/Code Projects/master/dm2026b/exp"

[1] "/Users/selewaut/Code Projects/master/dm2026b/kaggle.json"

In [7]:
PARAM <- list()
# reemplazar por su primer semilla
PARAM$semilla_primigenia <- 300089
PARAM$qsemillas <- 5    

PARAM$training_pct <- 70L  # entre  1L y 99L

# elegir SU dataset comentando/ descomentando
PARAM$dataset_nom <- file.path(DATA_DIR, "dataset_pequeno.csv")

In [8]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30), agrupa=clase_ternaria, seed=semilla)
#   crea una particion 70, 30

# modifies data object, no need to return.
particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}


In [9]:
ArbolEstimarGanancia <- function(semilla, training_pct, param_basicos) {
  # particiono estratificadamente el dataset
  particionar(dataset,
    division = c(training_pct, 100L -training_pct),
    agrupa = "clase_ternaria",
    seed = semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo <- rpart("clase_ternaria ~ .",
    data = dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval = 0,
    control = param_basicos
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion <- predict(modelo, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type = "prob"
  ) # type= "prob"  es que devuelva la probabilidad

  # prediccion es una matriz con TRES columnas,
  #  llamadas "BAJA+1", "BAJA+2"  y "CONTINUA"
  # cada columna es el vector de probabilidades


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test <- dataset[
    fold == 2,
    sum(ifelse(prediccion[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

  return(
    c( list("semilla" = semilla),
      param_basicos,
      list( "ganancia_test" = ganancia_test_normalizada )
     )
  )
}


In [ ]:
ArbolesMontecarlo <- function(semillas, param_basicos) {
  # Run seeds sequentially within the worker process
  salida <- lapply(semillas, function(s) {
    ArbolEstimarGanancia(
      semilla       = s,
      training_pct  = PARAM$training_pct,
      param_basicos = param_basicos
    )
  })

  return(salida)
}

In [11]:
# carpeta de trabajo
# por favor cambiar numero de experimento si se cambia el loop principal
experimento <- "HT29003"
dir.create(file.path(EXP_DIR, experimento), showWarnings=FALSE)
setwd(file.path(EXP_DIR, experimento))

In [12]:
# lectura del dataset
dataset <- fread(file.path(DATA_DIR, "dataset_pequeno.csv"))

# trabajo solo con los datos con clase, es decir 202107
dataset <- dataset[clase_ternaria != ""]

In [13]:
nrow(dataset)

[1] 164479

In [14]:
# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo
# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas)
PARAM$semillas


[1] 644857 218249 493747 655103 169327

In [15]:
file.exists(file.path(EXP_DIR, experimento,"gridsearch_detalle.txt"))

[1] TRUE

In [16]:
# genero la data.table donde van los resultados detallados del Grid Search
# un registro para cada combinacion de < semilla, parametros >

if(file.exists(file.path(EXP_DIR, experimento,"gridsearch_detalle.txt"))){
  tb_grid_search_detalle <- fread(file.path(EXP_DIR, experimento,"gridsearch_detalle.txt"))
}else{
  tb_grid_search_detalle <- data.table(
    semilla = integer(),
    cp = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket = integer(),
    ganancia_test = numeric()
  )
}

nrow( tb_grid_search_detalle )

[1] 336

In [17]:
# # ===============================================================
# # Grid Search for rpart - Parallel over hyper-parameters
# # Dataset ~160k rows | 1 seed (but compatible with multiple)
# # ===============================================================

# library(data.table)
# library(parallel)

# # ---------------------------------------------------------------
# # 1. Define a sensible grid for ~160k rows
# # ---------------------------------------------------------------
# grid <- CJ(
#   maxdepth  = c(6, 8, 10, 12, 14, 16, 20),
#   minsplit  = seq(150, 300, by = 20),
#   cp = c(-1)
#   )

# # Generate relative minbucket (classic + two alternatives)
# grid1 <- copy(grid)
# grid1[, minbucket := pmax(20L, as.integer(round(minsplit / 3)))]

# grid2 <- copy(grid)
# grid2[, minbucket := pmax(20L, as.integer(round(minsplit / 4)))]

# grid3 <- copy(grid)
# grid3[, minbucket := pmax(20L, as.integer(round(minsplit / 5)))]

# grid <- unique(rbind(grid1, grid2, grid3))
# setorder(grid, maxdepth, minsplit, minbucket, cp)

# cat("Total sensible combinations:", nrow(grid), "\n\n")
# cat("Total sensible combinations G1:", nrow(grid1), "\n\n")
# cat("Total sensible combinations G2:", nrow(grid2), "\n\n")
# cat("Total sensible combinations G3:", nrow(grid3), "\n\n")
# # grid <- unique(rbind(grid))

In [21]:
library(data.table)

# ---------------------------------------------------------------
# Expanded Grid Search (cp = -1 focus)
# ---------------------------------------------------------------
grid <- CJ(
  maxdepth  = c(4, 6, 8, 10, 12, 15, 20),
  minsplit  = c(15, 30, 60, 100, 150, 200, 300, 500, 1000),
  mb_ratio  = c(1/2, 1/3, 1/4, 1/6, 1/10), # Wider fraction selection
  cp        = -1
)

# Derive integer minbucket with a lower hard floor (5L instead of 20L)
grid[, minbucket := pmax(5L, as.integer(round(minsplit * mb_ratio)))]

# Drop temporary ratio column and remove duplicate rows caused by integer rounding
grid[, mb_ratio := NULL]
grid <- unique(grid)

# Sort logically
setorder(grid, maxdepth, minsplit, minbucket)

cat("Total unique parameter combinations:", nrow(grid), "\n")

Total unique parameter combinations: 287 


In [22]:
# ---------------------------------------------------------------
# 2. Load previous results (resume) - robust version
# ---------------------------------------------------------------

file_detalle <- file.path(EXP_DIR, experimento, "gridsearch_detalle.txt")

if (file.exists(file_detalle)) {
  tb_grid_search_detalle <- fread(file_detalle)
  cat("Loaded previous results:", nrow(tb_grid_search_detalle), "rows\n")
} else {
  # Create empty table with explicit schema
  tb_grid_search_detalle <- data.table(
    semilla       = integer(),
    maxdepth      = integer(),
    minsplit      = integer(),
    minbucket     = integer(),
    cp            = numeric(),
    ganancia_test = numeric()
  )
  cat("No previous results found. Starting fresh.\n")
}

# ---------- Dynamic & Fast Key Generator ----------
make_key <- function(dt) {
  if (nrow(dt) == 0L) return(character(0))
  
  # Include 'semilla' dynamically if present in the data.table
  cols <- intersect(c("semilla", "maxdepth", "minsplit", "minbucket", "cp"), names(dt))
  
  # Vectorized string conversion for maximum performance
  components <- lapply(cols, function(col) {
    if (col == "cp") {
      sprintf("%.8f", as.numeric(dt[[col]]))  # Prevents float serialization mismatches
    } else {
      as.integer(dt[[col]])
    }
  })
  
  do.call(paste, c(components, list(sep = "|")))
}

done_keys    <- make_key(tb_grid_search_detalle)
pending_keys <- make_key(grid)

# Efficient subsetting using set keys
grid_pending <- grid[!(pending_keys %in% done_keys)]

cat("Total combinations in grid :", nrow(grid), "\n")
cat("Already done               :", length(unique(done_keys)), "\n")
cat("Pending                    :", nrow(grid_pending), "\n\n")

Loaded previous results: 336 rows
Total combinations in grid : 287 
Already done               : 336 
Pending                    : 287 



In [24]:
library(flock)

In [ ]:
# check if experimetn is complete
if (nrow(grid_pending) == 0) {
  cat("Nothing to do. Grid search already complete.\n")
} else {

  # ---------------------------------------------------------------
  # 3. Function that evaluates ONE hyper-parameter combination
  # ---------------------------------------------------------------
  evaluar_una <- function(i) {
    log_file <- "progreso_experimento.txt"
    param_basicos <- as.list(grid_pending[i])
    # ArbolesMontecarlo works with 1 or many seeds
    ganancia_list <- ArbolesMontecarlo(PARAM$semillas, param_basicos)
    # Return as data.table
    res <- rbindlist(ganancia_list, use.names = TRUE, fill = TRUE)
    lock <- lock(log_file)
    cat(sprintf("[%s] Finalizada corrida: %s\n", Sys.time(), as.character(i)),
    file = log_file, append = TRUE) # nolint
    unlock(lock)

    # Explicitly return the result
    return(res)
  }

  # ---------------------------------------------------------------
  # 4. Parallel execution over combinations
  # ---------------------------------------------------------------
  ncores <- detectCores() - 8
  cat("Using", ncores, "cores for parallel grid search\n\n")

  resultados <- mclapply(
    X              = seq_len(nrow(grid_pending)),
    FUN            = evaluar_una,
    mc.cores       = ncores,
    mc.preschedule = FALSE      # better load balancing
  )

  # ---------------------------------------------------------------
  # 5. Combine & save
  # ---------------------------------------------------------------
  tb_nuevos <- rbindlist(resultados, use.names = TRUE, fill = TRUE)

  tb_grid_search_detalle <- rbindlist(
    list(tb_grid_search_detalle, tb_nuevos),
    use.names = TRUE,
    fill = TRUE
  )

  fwrite(tb_grid_search_detalle, "gridsearch_detalle.txt", sep = "\t")
  cat("\nFinished!\n")
  cat("Total rows now in gridsearch_detalle.txt:", nrow(tb_grid_search_detalle), "\n")
}

# ---------------------------------------------------------------
# 6. Quick summary of results (optional)
# ---------------------------------------------------------------
if (nrow(tb_grid_search_detalle) > 0) {
  resumen <- tb_grid_search_detalle[
    , .(
        ganancia_mean = mean(ganancia_test),
        ganancia_sd   = sd(ganancia_test),
        n             = .N
      ),
    by = .(maxdepth, minsplit, minbucket, cp)
  ][order(-ganancia_mean)]

  cat("\n===== Top 15 configurations =====\n")
  print(head(resumen, 15))
}

Using 10 cores for parallel grid search


Finished!
Total rows now in gridsearch_detalle.txt: 336 

===== Top 15 configurations =====
    maxdepth minsplit minbucket    cp ganancia_mean ganancia_sd     n
       <num>    <num>     <int> <num>         <num>       <num> <int>
 1:        8      270        90    -1     472333333          NA     1
 2:        6      170        34    -1     465916667          NA     1
 3:        6      150        38    -1     464666667          NA     1
 4:        8      270        68    -1     462333333          NA     1
 5:       10      290        58    -1     460666667          NA     1
 6:        6      170        42    -1     459833333          NA     1
 7:        6      190        38    -1     459833333          NA     1
 8:        6      210        42    -1     459833333          NA     1
 9:        8      290        97    -1     459333333          NA     1
10:        6      230        46    -1     457750000          NA     1
11:        6      230     

: 

Esta es la parte del código que usted debe expandir a TODOS los hiperparámetros de rpart,
<br>ya que actualmente apenas recorre  maxdepth y  minsplit  dejando fijos  cp=-0.5  y minbucket=5

In [ ]:

# # itero por los loops anidados para cada hiperparametro
# iter <- 0

# for (vmax_depth in c(4, 6, 8, 10, 12, 14)) {
#   for (vmin_split in c(1000, 800, 600, 400, 200, 100, 50, 20, 10)) {
#     # notar como se agrega

#     iter <- iter + 1
#     cat( iter, " " )
#     flush.console()
#     if( iter*PARAM$qsemillas < nrow(tb_grid_search_detalle)+1 ) next

#     # vminsplit  minima cantidad de registros en un nodo para hacer el split
#     param_basicos <- list(
#       "cp" = -0.5, # complejidad minima
#       "maxdepth" = vmax_depth, # profundidad máxima del arbol
#       "minsplit" = vmin_split, # tamaño minimo de nodo para hacer split
#       "minbucket" = 5 # minima cantidad de registros en una hoja
#     )

#     # Un solo llamado, con la semilla 17
#     ganancias <- ArbolesMontecarlo(PARAM$semillas, param_basicos)

#     # agrego a la tabla
#     tb_grid_search_detalle <- rbindlist(
#       list( tb_grid_search_detalle,
#             rbindlist(ganancias) )
#     )

#   }

#   # grabo cada vez TODA la tabla en el loop mas externo
#   fwrite( tb_grid_search_detalle,
#           file = "gridsearch_detalle.txt",
#           sep = "\t" )
# }


In [ ]:
# fwrite( tb_grid_search_detalle,
#    file = "gridsearch_detalle.txt",
#    sep = "\t"
# )

In [ ]:
# cantidad de registros de la tabla
nrow(tb_grid_search_detalle)

In [ ]:
# muestro la tabla
tb_grid_search_detalle

In [ ]:
# genero y grabo el resumen
tb_grid_search <- tb_grid_search_detalle[,
  list( "ganancia_mean" = mean(ganancia_test),
    "qty" = .N ),
  list( cp, maxdepth, minsplit, minbucket )
]


In [ ]:
# ordeno descendente por ganancia
setorder( tb_grid_search, -ganancia_mean )


In [ ]:
# veo los 10 mejores hiperparámetros
tb_grid_search[1:10]

In [ ]:
# genero un id a la tabla
tb_grid_search[, id := .I ]

fwrite( tb_grid_search,
  file = "gridsearch.txt",
  sep = "\t"
)


# 4.  Análisis de resultados de Grid Search

La salida de la corrida anterior queda en ~/buckets/b1/exp/HT2900  que corresponde a su Google Drive
<br>HT significa Hyperparameter Tuning
<br>El Grid Search es un método de fuerza bruta de un altísimo costo computacional.
<br>Queremos ver si es posible crear un algoritmo de optimización de hiperparámetros que se ahorre recorrer ciertas porciones muy malas del espacio de búsqueda. Algo del estilo “cada vez que pruebo una combinación de hiperparámetros donde  cp > 1 , la ganancia es muy mala, con lo cual ni vale la pena perder el tiempo explorando en esa region”


<br>Levante el archivo de salida gridsearch.txt  a una planilla tipo Excel y analícelo detenidamente
<br>Ordene por ganancia_mean descendente
<br>
<br>En la Planilla Colaborativa, Hoja  C3-GridSearch  cargue el mejor del ranking en la posición 1, el segundo en la 2, y el 5, 10, 50 y 100. Es decir debe cargr SEIS lineas en las celdas correspondientes a su nombre.
<br>
<br>Verifique que efectivamente está dado de alta en la competencia Kaggle  "Data Mining, Inicial 2026 B"  lo que debio haber hecho siguiendo el capítulo Arranque en Frio de El Libro de la Asignatura
<br>En la Planilla Colaborativa, Hoja  C3-GridSearch, utilizando el notebook  **src/arboles/z102_FinalTrain.ipynb**  haga el submit a Kaggle de cada una de las SEIS combinaciones de hiperparámetros y completa la columna Public Leaderboard

<br>
<br>El de mayor ganancia_mean  decimos que es el primero del ranking
En Zulip, correspondiente channel  #Tarea Hogar 02 , topic Analisis Grid Search   intente contestar estas preguntas:

* ¿Qué combinaciones de hiperparámetros poseen una ganancia muy buena?
* ¿Hay algun hiperparámetro que para cierto valor siempre genera una ganancia muy mala, a independientemente de lo que valgan los otros hiperparámetros ?
* ¿Que combinaciones de hiperparámetros es pésima y hubiera sido bueno ahorrarse esas corridas ?

( tiempo estimado 40 minutos, dificultad media )